In [ ]:
# repository setup runs in the next cell

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if Path('/kaggle/input').exists():
    repo_dir = Path('/kaggle/working/repo')
    if (repo_dir / '.git').exists():
        print('Kaggle repository exists. Updating main...')
        subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
    else:
        print('Kaggle environment detected. Cloning main...')
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', 'https://github.com/dvydinh/smpPrediction.git', str(repo_dir)], check=True)
    os.chdir(repo_dir)
    sys.path.insert(0, str(repo_dir))
    commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
    print(f'Repository commit {commit}')

os.system(f'{sys.executable} -m pip install -q -r requirements.txt')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from src.data_preprocessing import get_data_paths, load_and_preprocess_data
from src.feature_engineering import add_engineered_features
from src.feature_policy import select_production_features
from src.research_validation import run_pretest_validation, train_adaptive_model
from src import model_utils
from src import evaluation

DATA_ROOT, OUTPUT_DIR = get_data_paths()
print(f'Data root: {DATA_ROOT}')
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
df = load_and_preprocess_data(DATA_ROOT)

In [ ]:
df = add_engineered_features(df)

In [ ]:
feature_cols = select_production_features(df)
print(f'Feature count: {len(feature_cols)}')

RUN_FINAL_2026 = True
pretest_summary, pretest_metrics = run_pretest_validation(
    df, feature_cols,
    output_dir=str(OUTPUT_DIR / 'pretest'),
)


In [ ]:
model = None
final_executed = False
selected_features = feature_cols
if RUN_FINAL_2026:
    if not pretest_summary['target_met']:
        print('Pretest target is not met. The selected configuration will still receive the one-time 2026 test.')
    model, selected_features = train_adaptive_model(
        df, feature_cols,
        validation_summary=pretest_summary,
        output_dir=str(OUTPUT_DIR / 'models'),
        model_name='adaptive_window.pkl'
    )
    final_executed = True
else:
    print('Pretest complete. The 2026 test was not evaluated.')


In [ ]:
if final_executed:
    evaluation.evaluate_and_plot(
        model, df, selected_features,
        output_dir=str(OUTPUT_DIR / 'eval')
    )


In [ ]:
if final_executed:
    artifact_paths = {
        'model': OUTPUT_DIR / 'models' / 'adaptive_window.pkl',
        'metrics': OUTPUT_DIR / 'eval' / 'metrics.txt',
        'manifest': OUTPUT_DIR / 'eval' / 'model_manifest.json',
        'predictions': OUTPUT_DIR / 'eval' / 'predictions_2026.csv',
        'cycle metrics': OUTPUT_DIR / 'eval' / 'metrics_by_cycle.csv',
        'monthly metrics': OUTPUT_DIR / 'eval' / 'metrics_by_month.csv',
    }
else:
    artifact_paths = {
        'pretest metrics': OUTPUT_DIR / 'pretest' / 'pretest_metrics.csv',
        'pretest summary': OUTPUT_DIR / 'pretest' / 'pretest_summary.json',
    }
missing = [str(path) for path in artifact_paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing Kaggle artifacts: {missing}')

print('Artifacts remain in Kaggle output and will not be pushed to git.')
for name, path in artifact_paths.items():
    print(f'{name}: {path}')